In [7]:
# !python run_weekly_update.py # --test

In [1]:
import sqlite3
from pathlib import Path

DEFAULT_DB = "/mnt/d/forCoding_data/QuantFinance/plan_3-standardization_1/stock_data.db"

def verify_stock_db(db_path: str = DEFAULT_DB, sample_stocks: int = 20) -> dict:
    db_path = str(db_path)
    if not Path(db_path).exists():
        raise FileNotFoundError(f"数据库不存在: {db_path}")

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        cur = conn.cursor()

        tables = {"stocks", "stock_daily", "stock_weekly"}
        existing = {
            r["name"]
            for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
        }
        missing = sorted(tables - existing)
        if missing:
            raise RuntimeError(f"缺少数据表: {missing}")

        integrity = cur.execute("PRAGMA integrity_check").fetchone()[0]
        fk_issues = cur.execute("PRAGMA foreign_key_check").fetchall()

        report = {
            "db_path": db_path,
            "integrity_check": integrity,
            "foreign_key_issues_count": len(fk_issues),
            "tables": {},
            "sample": [],
        }

        def table_stats(table: str):
            total_rows = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

            if table == "stocks":
                distinct_codes = cur.execute("SELECT COUNT(DISTINCT code) FROM stocks").fetchone()[0]
                return {
                    "rows": total_rows,
                    "distinct_codes": distinct_codes,
                }

            distinct_codes = cur.execute(f"SELECT COUNT(DISTINCT stock_code) FROM {table}").fetchone()[0]
            min_date, max_date = cur.execute(
                f"SELECT MIN(date) AS min_date, MAX(date) AS max_date FROM {table}"
            ).fetchone()
            null_close = cur.execute(f"SELECT COUNT(*) FROM {table} WHERE close IS NULL").fetchone()[0]

            dup = cur.execute(
                f"""
                SELECT stock_code, date, COUNT(*) AS c
                FROM {table}
                GROUP BY stock_code, date
                HAVING c > 1
                LIMIT 5
                """
            ).fetchall()

            return {
                "rows": total_rows,
                "distinct_stock_codes": distinct_codes,
                "min_date": min_date,
                "max_date": max_date,
                "null_close_rows": null_close,
                "duplicate_key_examples": [dict(r) for r in dup],
            }

        report["tables"]["stocks"] = table_stats("stocks")
        report["tables"]["stock_daily"] = table_stats("stock_daily")
        report["tables"]["stock_weekly"] = table_stats("stock_weekly")

        sample_codes = [
            r["code"]
            for r in cur.execute(
                "SELECT code FROM stocks ORDER BY RANDOM() LIMIT ?",
                (int(sample_stocks),),
            ).fetchall()
        ]

        for code in sample_codes:
            d = cur.execute(
                "SELECT COUNT(*) AS n, MIN(date) AS min_date, MAX(date) AS max_date FROM stock_daily WHERE stock_code=?",
                (code,),
            ).fetchone()
            w = cur.execute(
                "SELECT COUNT(*) AS n, MIN(date) AS min_date, MAX(date) AS max_date FROM stock_weekly WHERE stock_code=?",
                (code,),
            ).fetchone()
            report["sample"].append(
                {
                    "code": code,
                    "daily_rows": d["n"],
                    "daily_min_date": d["min_date"],
                    "daily_max_date": d["max_date"],
                    "weekly_rows": w["n"],
                    "weekly_min_date": w["min_date"],
                    "weekly_max_date": w["max_date"],
                }
            )

        return report
    finally:
        conn.close()

def print_verify_report(report: dict):
    print(f"DB: {report['db_path']}")
    print(f"integrity_check: {report['integrity_check']}")
    print(f"foreign_key_issues_count: {report['foreign_key_issues_count']}")
    print("-" * 60)
    for name, s in report["tables"].items():
        print(f"[{name}]")
        for k, v in s.items():
            if k == "duplicate_key_examples":
                print(f"  {k}: {len(v)}")
                for ex in v:
                    print(f"    - {ex}")
            else:
                print(f"  {k}: {v}")
        print("-" * 60)
    print("[sample stocks]")
    for row in report["sample"]:
        print(
            f"{row['code']} | "
            f"D {row['daily_rows']} rows {row['daily_min_date']}..{row['daily_max_date']} | "
            f"W {row['weekly_rows']} rows {row['weekly_min_date']}..{row['weekly_max_date']}"
        )

if __name__ == "__main__":
    r = verify_stock_db(DEFAULT_DB, sample_stocks=15)
    print_verify_report(r)

DB: /mnt/d/forCoding_data/QuantFinance/plan_3-standardization_1/stock_data.db
integrity_check: ok
foreign_key_issues_count: 0
------------------------------------------------------------
[stocks]
  rows: 5195
  distinct_codes: 5195
------------------------------------------------------------
[stock_daily]
  rows: 1563754
  distinct_stock_codes: 5195
  min_date: 2025-01-02
  max_date: 2026-04-08
  null_close_rows: 0
  duplicate_key_examples: 0
------------------------------------------------------------
[stock_weekly]
  rows: 334298
  distinct_stock_codes: 5194
  min_date: 2025-01-03
  max_date: 2026-04-03
  null_close_rows: 0
  duplicate_key_examples: 0
------------------------------------------------------------
[sample stocks]
sh.603602 | D 304 rows 2025-01-02..2026-04-08 | W 65 rows 2025-01-03..2026-04-03
sz.002082 | D 304 rows 2025-01-02..2026-04-08 | W 65 rows 2025-01-03..2026-04-03
sz.300897 | D 304 rows 2025-01-02..2026-04-08 | W 65 rows 2025-01-03..2026-04-03
sz.002204 | D 304 